<a href="https://colab.research.google.com/github/Davidson-Open-University/phy107-labs/blob/main/notebooks/PHY107-Lab1-Measurement-Variables-and-Fair-Tests.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY 107 Lab 1: Measurement, Variables and Fair Tests

**Unit 1: Introduction to practical physics and mechanics** · General Practical Physics I · Davidson Open University

**In this lab you will:**
- take repeated readings and summarise them with a mean and a spread;
- round a result to a sensible number of significant figures;
- convert units and calculate weight from mass;
- identify the independent, dependent and controlled variables in an experiment.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the online lab and the examples, and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short analysis function and test it on data.
- **Part D – Your own data and reflection** (about 10 minutes): analyse readings you took in the online lab (or at home), and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `phy107-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

**Honest data:** keep your own readings separate from the example data, and label every simulated or supplied reading as such. Never invent readings.

*This lab is practice to help you learn and prepare for the unit assignment (lab report). It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False, dp=6):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), dp) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i, False, dp) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i, False, dp) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k, False, dp) + '=' + _norm(v, False, dp) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), dp) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('phy107|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False, spec.get('dp', 6))
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False), spec.get('dp', 6))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '1.1': {'h': '6b11f3a4a1a8cf31', 'hint': 'Use np.mean(times), or sum(times)/len(times).', 'ok': 'The mean is the best single value from repeated readings.', 'dp': 2},
    '1.2': {'h': '1ff5e31134b7c058', 'hint': 'spread = max(times) - min(times).', 'ok': 'Half the spread is a quick estimate of the uncertainty: ±0.2 s here.', 'dp': 1},
    '1.3': {'h': 'de05e321b4daebd6', 'hint': 'First 350 g = 0.350 kg (divide by 1000), then W = m × 9.81.', 'ok': 'A balance reads mass; a force gauge would read this weight.', 'dp': 2},
    '1.4': {'h': 'f2f9887a9b91697d', 'hint': 'The independent variable is the one YOU change. The dependent variable is the one you MEASURE. Controlled variables are kept the same.', 'ok': 'Naming variables clearly makes an experiment repeatable.'},
    '1.5': {'h': '0b1cfe5565c8ea39', 'hint': 't = np.sqrt(2 * d / g) with d = 0.20 and g = 9.81.', 'ok': 'About 0.2 s is a typical human reaction time.', 'dp': 3},
    '1.6': {'h': '98af9bf6e8f4485a', 'hint': 'mean = sum(readings) / len(readings); half spread = (max(readings) - min(readings)) / 2; return both in a tuple.', 'ok': 'Your summary tool works on every test set.', 'dp': 2, 'args': [([50.0, 50.5, 50.0],), ([12.4, 12.6, 12.5, 12.7, 12.3],), ([3.0, 3.0, 3.0],), ([0.18, 0.21, 0.19, 0.22],)]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
### 🔬 Online lab for this unit (free open educational resources)
- **[PhET Forces and Motion: Basics](https://phet.colorado.edu/en/simulations/forces-and-motion-basics)** – push a box with balanced and unbalanced forces and watch the motion (screens: Net Force, Motion).

Open the online lab in a new tab, try it for a few minutes, then come back here. **Label every reading you take there as "simulated".**

**In the online lab:** on the *Net Force* screen, put equal teams on each side. *Predict first*: will the cart move? Then add one more person on one side. Write down what changes and what you kept the same. That is the start of a **fair test**: change one thing, measure its effect, keep everything else the same.

**Repeated readings.** One reading can be wrong. Physicists repeat a measurement and report the **mean** (average) and the **spread** (largest minus smallest). Run the cell: it uses the illustrative block lengths from Unit 1 (50.0, 50.5 and 50.0 mm).

In [ ]:
import numpy as np
readings = [50.0, 50.5, 50.0]          # mm (illustrative data from the unit, not your own)
mean = np.mean(readings)
spread = max(readings) - min(readings)
print(f'Mean = {mean:.4f} mm  ->  report about {mean:.1f} mm')
print(f'Spread = {spread:.1f} mm')

**Why round?** Reporting 50.1666667 mm claims a precision the ruler cannot give. A ruler read to 0.5 mm supports one decimal place at most, so we report **50.2 mm**. `round(x, 1)` rounds to one decimal place.

**Weight is a force.** A balance gives **mass** in kg; the **weight** is W = mg, in newtons (N), with g = 9.81 m/s².

In [ ]:
m = 0.350          # kg
g = 9.81           # m/s²
W = m * g
print(f'W = {W:.4f} N  ->  {W:.3g} N to 3 significant figures')

## Part B – Practise (about 20 minutes)
Replace each `...`, run the cell, then run the **check** cell under it. Answers are checked to the number of decimal places shown in the question.

**B1.** A student times the same trolley run five times: 12.4, 12.6, 12.5, 12.7 and 12.3 s. Find the **mean** time (2 decimal places).

In [ ]:
times = [12.4, 12.6, 12.5, 12.7, 12.3]   # seconds (illustrative)

In [ ]:
answer_1_1 = ...

In [ ]:
check('1.1', answer_1_1)

In [ ]:
answer_1_2 = ...   # the spread (largest minus smallest), 1 decimal place

In [ ]:
check('1.2', answer_1_2)

**B3.** Convert 350 g to kilograms and find the weight in newtons (g = 9.81 m/s²). Give the weight to 2 decimal places.

In [ ]:
answer_1_3 = ...

In [ ]:
check('1.3', answer_1_3)

**B4. Variables.** A trolley is released from different heights on a ramp, and the time to travel a fixed 1.0 m is measured. The same trolley and release method are used every time. Choose letters:

- a) travel time  b) release height  c) the trolley and release method

Store `(independent, dependent, controlled)` as a tuple of letters, for example `('a', 'b', 'c')`.

In [ ]:
answer_1_4 = ...

In [ ]:
check('1.4', answer_1_4)

**B5. Reaction time by ruler drop.** A partner drops a ruler and you catch it after it falls a distance d. Free fall gives d = ½gt², so t = √(2d/g). If the ruler falls 0.20 m, what is your reaction time in seconds (3 decimal places)? *Use* `np.sqrt`.

In [ ]:
answer_1_5 = ...

In [ ]:
check('1.5', answer_1_5)

## Part C – Build a tool (about 20 minutes)
Complete `summarise(readings)`. It should return a tuple **(mean, uncertainty)**, where the uncertainty is **half the spread**, both rounded to 2 decimal places. You will reuse this tool in later labs.

In [ ]:
def summarise(readings):
    # YOUR CODE: return (mean rounded to 2 dp, half the spread rounded to 2 dp)
    ...

print(summarise([50.0, 50.5, 50.0]))

In [ ]:
check('1.6', summarise)

## Part D – Your own data and reflection (about 10 minutes)
**D1. Measure your reaction time (safe home task, prepares you for the Unit 1 assignment).** With a partner, drop a 30 cm ruler between your open finger and thumb and catch it. Record the distance it fell (in metres) five times. If you have no partner, use the Net Force screen in the online lab instead: record five settings and the motion you saw, labelled "simulated". Enter your readings, then use your tools.

In [ ]:
my_drops = [...]          # metres, your own readings, e.g. [0.18, 0.21, ...]
# reaction_times = [np.sqrt(2 * d / 9.81) for d in my_drops]
# print(summarise(reaction_times))

**D2. Reflect.** Answer in one or two sentences each:
1. Why is one reading not enough?
2. Five identical readings: do they prove the ruler has no zero error? Why or why not?
3. Which variables did you control in D1?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `phy107-labs` repository → commit message *"Lab 1 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.